# =============================================================================
# ABC (Abundance-Biomass Comparison) Analysis for Mormyridae in Niger River
# Author: Souleymane Maman Nouri Souley
# Description: Computes ABC_W, DAP, Log_e(SEP) and generates k-dominance curves for spatial (station) and temporal (month) groupings.
# =============================================================================

# =============================================================================
# Loading the packages
# =============================================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

In [ ]:
INPUT_FILE = '[YOUR DATA PATH]'
OUTPUT_DIR = '[RESULTS]'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# 2. LOAD AND CLEAN DATA
# =============================================================================

In [ ]:
df = pd.read_csv(INPUT_FILE, sep=',')
df.columns = df.columns.str.strip()
df.rename(columns={
    'Espèces': 'Species',
    'Lt (cm)': 'Total_Length',
    'Ls (cm)': 'Standard_Length',
    'Pt (g)': 'Weight'
}, inplace=True)

# Remove any rows with missing weight (biomass) or species
df = df.dropna(subset=['Species', 'Weight'])
print(f"Total individuals: {len(df)}")

# =============================================================================
# 3. CORE ABC FUNCTION
# =============================================================================

In [ ]:
def compute_abc_indices(group_df, group_name='Group'):
    """
    Compute ABC_W, DAP, and Log_e(SEP) for a given grouped DataFrame.
    Returns a dict with the indices and the k-dominance curve data.
    """
    # Aggregate abundance (count) and biomass (sum of weight) per species
    agg = group_df.groupby('Species').agg(
        Abundance=('Weight', 'count'),
        Biomass=('Weight', 'sum')
    ).reset_index()
    agg = agg[agg['Abundance'] > 0].copy()
    total_abund = agg['Abundance'].sum()
    total_biom = agg['Biomass'].sum()

    # Sort by abundance (descending) – this sets the rank order for ABC
    sorted_abund = agg.sort_values('Abundance', ascending=False).reset_index(drop=True)
    S = len(sorted_abund)

    # Cumulative abundance %
    sorted_abund['Cum_Abund_pct'] = 100 * sorted_abund['Abundance'].cumsum() / total_abund

    # Cumulative biomass % in the SAME order (abundance rank)
    cum_biom = 0
    cum_biom_pct_list = []
    for _, row in sorted_abund.iterrows():
        cum_biom += row['Biomass']
        cum_biom_pct_list.append(100 * cum_biom / total_biom)
    sorted_abund['Cum_Biom_pct'] = cum_biom_pct_list

    # ABC_W = mean difference at each rank
    diff = sorted_abund['Cum_Biom_pct'] - sorted_abund['Cum_Abund_pct']
    ABC_W = diff.mean()

    # DAP: Area between curves over ln(rank)
    ranks = np.arange(1, S + 1)
    ln_ranks = np.log(ranks)
    area_abund = np.trapz(sorted_abund['Cum_Abund_pct'], ln_ranks)
    area_biom = np.trapz(sorted_abund['Cum_Biom_pct'], ln_ranks)
    DAP = (area_biom - area_abund) / np.log(S) if S > 1 else 0.0

    # SEP: Shannon diversity on biomass vs abundance
    p_abund = agg['Abundance'] / total_abund
    p_biom = agg['Biomass'] / total_biom
    # Use natural log for both (base cancels out in the ratio)
    H_abund = entropy(p_abund, base=np.e)
    H_biom = entropy(p_biom, base=np.e)
    SEP = H_biom / H_abund if H_abund > 0 else 1.0
    log_SEP = np.log(SEP)

    # Prepare k-dominance curve data for plotting
    curve_data = {
        'ln_rank': ln_ranks,
        'Cum_Abund_pct': sorted_abund['Cum_Abund_pct'],
        'Cum_Biom_pct': sorted_abund['Cum_Biom_pct'],
        'S': S
    }

    return {
        'ABC_W': ABC_W,
        'DAP': DAP,
        'log_SEP': log_SEP,
        'curve_data': curve_data
    }

# =============================================================================
# 4. SPATIAL ANALYSIS (Pooled by Station)
# =============================================================================

In [ ]:
stations = df['Station'].unique()
spatial_results = []
for st in stations:
    sub = df[df['Station'] == st]
    res = compute_abc_indices(sub, st)
    spatial_results.append({
        'Station': st,
        'ABC_W': res['ABC_W'],
        'DAP': res['DAP'],
        'log_SEP': res['log_SEP'],
        'curve_data': res['curve_data']
    })

# Convert to DataFrame and sort by ABC_W (Gamkalley highest = unstressed)
spatial_df = pd.DataFrame(spatial_results).sort_values('ABC_W', ascending=False)
print("\n=== SPATIAL ABC INDICES (pooled over months) ===")
print(spatial_df[['Station', 'ABC_W', 'DAP', 'log_SEP']].to_string(index=False))

# Save to CSV
spatial_df[['Station', 'ABC_W', 'DAP', 'log_SEP']].to_csv(
    os.path.join(OUTPUT_DIR, 'Table_ABC_Spatial.csv'), index=False
)

# =============================================================================
# 5. TEMPORAL ANALYSIS (Pooled by Month)
# =============================================================================

In [ ]:
months = ['August', 'September', 'October', 'November']
temporal_results = []
for mo in months:
    sub = df[df['Months'] == mo]
    res = compute_abc_indices(sub, mo)
    temporal_results.append({
        'Month': mo,
        'ABC_W': res['ABC_W'],
        'DAP': res['DAP'],
        'log_SEP': res['log_SEP'],
        'curve_data': res['curve_data']
    })

temporal_df = pd.DataFrame(temporal_results).sort_values('Month', key=lambda x: x.map(
    {'August': 0, 'September': 1, 'October': 2, 'November': 3}
))
print("\n=== TEMPORAL ABC INDICES (pooled over stations) ===")
print(temporal_df[['Month', 'ABC_W', 'DAP', 'log_SEP']].to_string(index=False))

# Save to CSV
temporal_df[['Month', 'ABC_W', 'DAP', 'log_SEP']].to_csv(
    os.path.join(OUTPUT_DIR, 'Table_ABC_Temporal.csv'), index=False
)

# =============================================================================
# 6. GENERATE K-DOMINANCE CURVES (Publication ready)
# =============================================================================

In [ ]:
def plot_kdominance(curve_data, title, filename, dpi=300):
    plt.figure(figsize=(5.0, 4.0), dpi=dpi)
    plt.plot(curve_data['ln_rank'], curve_data['Cum_Abund_pct'],
             'k--', linewidth=2, label='Abundance')
    plt.plot(curve_data['ln_rank'], curve_data['Cum_Biom_pct'],
             'k-', linewidth=2, label='Biomass')
    plt.xlabel('ln(rank)', fontsize=12)
    plt.ylabel('Cumulative %', fontsize=12)
    plt.title(title, fontsize=11, pad=10)
    plt.legend(loc='best', frameon=False)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    # Save as PDF (vector) and PNG (high res)
    plt.savefig(os.path.join(OUTPUT_DIR, f'{filename}.pdf'), format='pdf', bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR, f'{filename}.png'), dpi=300, bbox_inches='tight')
    plt.close()

# Spatial plots
for res in spatial_results:
    st = res['Station']
    plot_kdominance(
        res['curve_data'],
        f'Station: {st}',
        f'kdom_station_{st.replace(" ", "_")}'
    )

# Temporal plots
for res in temporal_results:
    mo = res['Month']
    plot_kdominance(
        res['curve_data'],
        f'Month: {mo}',
        f'kdom_month_{mo}'
    )

print(f"\nAll figures and tables saved to '{OUTPUT_DIR}/'")

# =============================================================================
# 7. PRINT SUMMARY FOR MANUSCRIPT (Copy-paste ready)
# =============================================================================

In [ ]:
print("\n" + "="*60)
print("COPY THESE VALUES INTO YOUR MANUSCRIPT TABLES:")
print("="*60)

print("\n--- TABLE 4 (Spatial ABC indices) ---")
print(spatial_df[['Station', 'ABC_W', 'DAP', 'log_SEP']].to_string(index=False, float_format="%+.6f"))

print("\n--- TABLE 5 (Monthly ABC indices) ---")
print(temporal_df[['Month', 'ABC_W', 'DAP', 'log_SEP']].to_string(index=False, float_format="%+.6f"))

print("\n--- K-DOMINANCE CURVE FILES GENERATED ---")
print("PDF and PNG files are in the 'results' folder.")